In [ ]:
%load_ext autoreload
%autoreload 2

### Update with ESCAPE-seq data

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "CLSBN26":
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find CLSBN26 project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "Data"
SRC_DIR = PROJECT_ROOT / "src"
MODEL_CHECKPOINTS_DIR = PROJECT_ROOT / "Model_Checkpoints"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("PROJECT_ROOT in sys.path:", str(PROJECT_ROOT) in sys.path)

In [ ]:
# imports
from src.general_utils import *
from src.get_feature_data import *
from src.Model.mhcprime import *
from src.model_utils import *
from src.general_utils import *
from src.get_feature_data import *
from src.pepcl import *
from src.mhcprime_inference import *
from src.log_odds import *
from src.plotting import *

# misc
import warnings
warnings.filterwarnings("ignore")
device="cuda"

In [ ]:
# change gpu
# GPU_ID = 0
# GPU_ID = 1
GPU_ID = 2
# GPU_ID = 3

torch.cuda.set_device(GPU_ID)
print(f"Using GPU: {torch.cuda.get_device_name(GPU_ID)} (ID: {torch.cuda.current_device()})")

Using GPU: NVIDIA H100 80GB HBM3 (ID: 2)


### Get datasets

#### Format pseudosequences and get feature data

In [ ]:
# format pseudosequences
from Data.mhc_pseudo_seqs import mhc_seq_dict
mhc_seq_dict={k.removeprefix("HLA-").replace("*","").replace(":",""):v for k,v in mhc_seq_dict.items()}

In [ ]:
# get feature table and processed features
aa_features_table_ext, descriptor_groups=load_feature_data()
processed_feature_table, corr_info = process_aa_feature_table(aa_features_table_ext)
feature_names = list(processed_feature_table.columns)  # pass to FeatureProjectionEncoder

#### MS/ESCAPE-seq train and test sets

In [ ]:
# ms train and test sets
datasets_dir = DATA_DIR / "Training_Datasets" / "Processed" / "MS_Train_Test_Sets"
ms_train_data=pd.read_csv(f"{datasets_dir}/ms_train_data.csv", index_col=0)
ms_test_data=pd.read_csv(f"{datasets_dir}/ms_test_data.csv", index_col=0)

In [ ]:
# escape seq sets
escape_seq_datasets_dir = DATA_DIR / "Training_Datasets" / "Processed" / "ESCAPE_seq"
oncogene_mut_wt_df=pd.read_csv(f"{escape_seq_datasets_dir}/oncogene_mut_wt_df.csv", index_col=0)
escapeseq_reported_neoantigens=pd.read_csv(f"{escape_seq_datasets_dir}/escapeseq_reported_neoantigens.csv", index_col=0)
escapeseq_reported_neoantigens_expanded=pd.read_csv(f"{escape_seq_datasets_dir}/escapeseq_reported_neoantigens_expanded.csv", index_col=0)

In [ ]:
# ligand mhc atlas set
ligand_mhc_atlas_output_dir=f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Ligand_MHC_Atlas"
ligand_mhc_atlas_cancer_test_set_for_escape_seq=pd.read_csv(f"{ligand_mhc_atlas_output_dir}/ligand_mhc_atlas_cancer_test_set_for_escape_seq.csv", index_col=0)

In [ ]:
# create version without reported neoantigens
oncogene_mut_wt_df.seq=oncogene_mut_wt_df.seq.str.replace("$", "")
escapeseq_reported_neoantigens_expanded.seq=escapeseq_reported_neoantigens_expanded.seq.str.replace("$", "")
oncogene_mut_wt_df=create_seq_allele_column(oncogene_mut_wt_df)
escapeseq_reported_neoantigens_expanded=create_seq_allele_column(escapeseq_reported_neoantigens_expanded)
seq_overlap=df_col_set_intersection(oncogene_mut_wt_df, escapeseq_reported_neoantigens_expanded, "seq", print_output=True, return_output=True)
oncogene_mut_wt_df_cleaned=oncogene_mut_wt_df[~oncogene_mut_wt_df.seq.isin(seq_overlap)]
oncogene_mut_wt_df_cleaned_dedup=oncogene_mut_wt_df_cleaned.drop_duplicates(subset=["seq_allele"])

20


In [ ]:
# merge to create train set
oncogene_mut_wt_df_cleaned_dedup.rename(columns={"escore": "ba_value"}, inplace=True)
oncogene_mut_wt_df_cleaned_dedup["domain_id"]=1
ms_train_data["domain_id"]=0
ms_escape_seq_train_data=pd.concat([ms_train_data, oncogene_mut_wt_df_cleaned_dedup])
ms_escape_seq_train_data.reset_index(inplace=True, drop=True)
print_data_stats(ms_escape_seq_train_data)

Total: 38,062,973
Pos: 752,461 | 1.98%
Neg: 37,310,512 | 98.02%
N alleles: 197


In [ ]:
# remove X containing seqs and A68 sequences
def filter_sequences_without_x(df, seq_col='seq'):
    return df[~df[seq_col].str.contains('X', na=False)].reset_index(drop=True)

ms_escape_seq_train_data_noX=filter_sequences_without_x(ms_escape_seq_train_data)
ms_escape_seq_train_data_noX_noA68=ms_escape_seq_train_data_noX.query("allele != 'A6801'")
ms_escape_seq_train_data_noX_noA68.reset_index(inplace=True, drop=True)
print_data_stats(ms_escape_seq_train_data_noX_noA68)

Total: 37,404,215
Pos: 739,533 | 1.98%
Neg: 36,664,682 | 98.02%
N alleles: 196


In [ ]:
# match alleles that are present in ms (for new alleles, its fine to keep them in)
ms_escape_seq_train_data_noX_noA68["seq_len"]=ms_escape_seq_train_data_noX_noA68.seq.str.replace("$", "").apply(len)
ms_escape_seq_train_data_noX_noA68_matched_allele=ms_escape_seq_train_data_noX_noA68[ms_escape_seq_train_data_noX_noA68.allele.isin(oncogene_mut_wt_df_cleaned_dedup.allele.unique().tolist())].query("seq_len == 9")
ms_escape_seq_train_data_noX_noA68_matched_allele.reset_index(inplace=True, drop=True)
print_data_stats(ms_escape_seq_train_data_noX_noA68_matched_allele)

Total: 12,340,124
Pos: 248,092 | 2.01%
Neg: 12,092,032 | 97.99%
N alleles: 49


In [ ]:
# full process
ms_escape_seq_train_data_noX_noA68_matched_allele=full_process_data(ms_escape_seq_train_data_noX_noA68_matched_allele, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)

escapeseq_reported_neoantigens=full_process_data(escapeseq_reported_neoantigens, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)

escapeseq_reported_neoantigens_expanded=full_process_data(escapeseq_reported_neoantigens_expanded, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)

#### Background datasets

In [ ]:
%%time
# for ranking
chp_background_10k_per_allele_x_len_processed=pd.read_csv(f"{DATA_DIR}/chp_background_10k_per_allele_x_len_processed.csv.gz")

CPU times: user 29 s, sys: 4 s, total: 33 s
Wall time: 32.9 s


### Load MHCPrime

In [ ]:
# setup tokenizer
tokenizer = AminoAcidTokenizer()

In [ ]:
# set base model params
base_model_params={
        'vocab_size': tokenizer.vocab_size,
        'embed_dim': 480,
        'fc_hidden_dims': [240, 120], 
        'fc_dropout': 0.1,
        'n_heads': 16,
        'max_peptide_len': 34,
        'max_mhc_len': 34,
        "n_peptide_layers": 2,
        "n_mhc_layers": 1,
        'n_concat_layers': 3,
        'add_pos_enc': True,
        'use_segment_embedder': True,
        'dim_feedforward_multiplier': 1,
        'debugging': False,
        'fc_output_dim': 1,
        'pooling': 'cls', # cls, mean, min, max, attn, for sequence ffnn head
        "ffnn_use_layernorm": False, 
        'use_feature_table': True,
        "feature_table": processed_feature_table,
        "sequence_embedder_feature_names": feature_names,
        'sequence_embedder_feature_mode': "concat",  # replace or concat
        'sequence_embedder_tokenizer': tokenizer,
        "peptide_start_pos": 1, 
        "mhc_start_pos": 35, 
        "pos_enc_type": "sincos", 
        "seed": 42,

        # peptide conformer
        "add_peptide_conformer": True, 
        "peptide_conformer_num_layers": 2,
        "peptide_conformer_n_heads": 16,
        "peptide_conformer_conv_kernel": 5,
        "peptide_conformer_dropout": 0.0,

        # mhc conformer
        "add_mhc_conformer": False, 
        "mhc_conformer_num_layers": 2,
        "mhc_conformer_n_heads": 16,
        "mhc_conformer_conv_kernel": 5,
        "mhc_conformer_dropout": 0.0,

        # fusion conformer parameters
        "add_fusion_conformer": True, 
        "fusion_conformer_num_layers": 1,
        "fusion_conformer_n_heads": 16,
        "fusion_conformer_conv_kernel": 7,
        "fusion_conformer_dropout": 0.0,
        "fusion_conformer_mask_bos": True,
        "ff_pre_post_swiglu": False,

        # legacy init
        "legacy_init_compat": True,
}

In [ ]:
%%time
# load base model (optionally, can load model_params directly)
# run_name = "MHCPrime_Base"
# run_dir = MODEL_CHECKPOINTS_DIR / run_name
# model_dir = run_dir / "Checkpoints"

# mhcprime=init_model(base_model_params, print_check=False, print_params=True)
# mhcprime.load_state_dict(torch.load(f"{model_dir}/model_final.pt"))
# mhcprime.eval()
# print("Model loaded from checkpoint...")
# clear_all_gpu_memory()

base_model_dir = MODEL_CHECKPOINTS_DIR / "MHCPrime_Base" / "Checkpoints"
ckpt_path = base_model_dir / "model_final.pt"

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

clear_all_gpu_memory()

mhcprime = init_model(
    base_model_params,
    print_check=False,
    print_params=True,
)

state_dict = torch.load(
    ckpt_path,
    map_location="cpu",
)

mhcprime.load_state_dict(state_dict, strict=True)
mhcprime.to(device)
mhcprime.eval()

print("Model loaded from checkpoint...")
print(f"Checkpoint: {ckpt_path}")
print(f"Model device: {next(mhcprime.parameters()).device}")


All GPU memory has been cleared.
Current GPU memory allocated: 0.00 GB
Current GPU memory reserved: 0.00 GB
[FeatureProjectionEncoder] Warning: no features found for tokens: ['$', '*', '<BOS>']
Number of trainable modules: 202
Number of trainable parameters: 25751991
Model loaded successfully...
Model loaded from checkpoint...
Checkpoint: /home/pchati/PepCL_Manuscript/Model_Checkpoints/MHCPrime_Base/Checkpoints/model_final.pt
Model device: cuda:0
CPU times: user 792 ms, sys: 494 ms, total: 1.29 s
Wall time: 1.3 s


### PepCL updating

#### Run PepCL

In [ ]:
# for classification based training
pepcl_base_params_cls=dict(
            tokenizer=tokenizer,
            device="cuda",
            steps = 200,
            output_dir = None,
            save_every_n_steps = 2000,
            seed = 42,
            num_workers = 16,
            collate_fn = collate_fn,
            num_ms_pos = 2000,
            num_nonms_pos = 2000,
            ms_neg_ratio = 1.0,
            nonms_neg_ratio = 1.0,
            auto_calibrate = True,
            calibration_steps = 1,
            ms_batch_size = 3072,
            nonms_batch_size = 3072,
            encoder_lr = 2e-4,
            decoder_lr = 2e-4,
            weight_decay = 0.01,
            warmup_ratio = 0.1,
            update_domain_embeddings = False,
            primary_task_id = 1,
            sequential_backprop = True,
            ms_loss_type = "logsmoothap",
            ms_loss_alpha = 10.0,
            ms_lambda = 1.0,
            nonms_loss_type = "logsmoothap",
            nonms_loss_alpha = 10.0,
            ft_lambda = 1.0,
            dr_lambda = 1e-5,
            dr_loss_type = "mse"
)

# for continuous value training
pepcl_base_params_con=dict(
            tokenizer=tokenizer,
            device="cuda",
            steps = 200,
            output_dir = None,
            save_every_n_steps = 2000,
            seed = 42,
            num_workers = 16,
            collate_fn = collate_fn,
            num_ms_pos = 2000,
            num_nonms_pos = 2000,
            ms_neg_ratio = 1.0,
            nonms_neg_ratio = 1.0,
            auto_calibrate = True,
            calibration_steps = 1,
            ms_batch_size = 3072,
            nonms_batch_size = 3072,
            encoder_lr = 2e-4,
            decoder_lr = 2e-4,
            weight_decay = 0.01,
            warmup_ratio = 0.1,
            update_domain_embeddings = False,
            primary_task_id = 1,
            sequential_backprop = True,
            ms_loss_type = "logsmoothap",
            ms_loss_alpha = 10.0,
            ms_lambda = 1.0,
            nonms_loss_type = "spearman",
            nonms_loss_alpha = 0.01,
            ft_lambda = 1.0,
            dr_lambda = 1e-5,
            dr_loss_type = "mse",
            continuous_target_col="ba_value",
            continuous_ft_sampling_mode="allele_aware",
)

In [ ]:
# init models
escape_seq_neoant_ft_cls = copy.deepcopy(mhcprime) # classification
escape_seq_neoant_pepcl_cls = copy.deepcopy(mhcprime) # classification
escape_seq_neoant_ft_con = copy.deepcopy(mhcprime) # continuous value
escape_seq_neoant_pepcl_con = copy.deepcopy(mhcprime) # continuous value
clear_all_gpu_memory()

All GPU memory has been cleared.
Current GPU memory allocated: 1.61 GB
Current GPU memory reserved: 4.77 GB


In [ ]:
%%time
# run updated pepcl function
pepcl=PepCL(
            model=escape_seq_neoant_ft_cls,
            train_df=ms_escape_seq_train_data_noX_noA68_matched_allele,
            use_replay = False,
            **pepcl_base_params_cls,
)
training_stats=pepcl.train()

[PepCL] Auto-calibrated sampling:
  MS: alleles=44, A_eff=19.80, balance=0.450, target_pos/cycle=1536, q_pos median=28.0 [18, 131], reuse median=0.0080, reuse max=0.0323, neg_ratio=1.0
  FT: alleles=49, A_eff=29.05, balance=0.593, target_pos/cycle=1536, q_pos median=28.0 [6, 69], reuse median=0.2478, reuse max=1.0000, neg_ratio=1.0



PepCL: 100%|██████████| 200/200 [01:07<00:00,  2.95it/s, MS=0.000000, FT=0.000303, DR=0.000000]


CPU times: user 1min 19s, sys: 13.9 s, total: 1min 33s
Wall time: 1min 33s


In [ ]:
%%time
# run updated pepcl function
pepcl=PepCL(
            model=escape_seq_neoant_pepcl_cls,
            train_df=ms_escape_seq_train_data_noX_noA68_matched_allele,
            use_replay = True,
            **pepcl_base_params_cls,
)
training_stats=pepcl.train()

[PepCL] Auto-calibrated sampling:
  MS: alleles=44, A_eff=19.80, balance=0.450, target_pos/cycle=1536, q_pos median=28.0 [18, 131], reuse median=0.0080, reuse max=0.0323, neg_ratio=1.0
  FT: alleles=49, A_eff=29.05, balance=0.593, target_pos/cycle=1536, q_pos median=28.0 [6, 69], reuse median=0.2478, reuse max=1.0000, neg_ratio=1.0



PepCL: 100%|██████████| 200/200 [02:37<00:00,  1.27it/s, MS=0.010883, FT=0.001458, DR=0.005917]


CPU times: user 2min 45s, sys: 21.8 s, total: 3min 7s
Wall time: 3min 4s


In [ ]:
%%time
# run updated pepcl function
pepcl=PepCL(
            model=escape_seq_neoant_ft_con,
            train_df=ms_escape_seq_train_data_noX_noA68_matched_allele,
            use_replay = False,
            **pepcl_base_params_con,
)
training_stats=pepcl.train()

[PepCL] Auto-calibrated sampling:
  MS: alleles=44, A_eff=19.80, balance=0.450, target_pos/cycle=1536, q_pos median=28.0 [18, 131], reuse median=0.0080, reuse max=0.0323, neg_ratio=1.0
  FT: alleles=49, A_eff=29.05, balance=0.593, target_pos/cycle=1536, q_pos median=28.0 [6, 69], reuse median=0.2478, reuse max=1.0000, neg_ratio=1.0



PepCL: 100%|██████████| 200/200 [01:08<00:00,  2.93it/s, MS=0.000000, FT=0.066714, DR=0.000000]


CPU times: user 1min 20s, sys: 16.3 s, total: 1min 36s
Wall time: 1min 35s


In [ ]:
%%time
# run updated pepcl function
pepcl=PepCL(
            model=escape_seq_neoant_pepcl_con,
            train_df=ms_escape_seq_train_data_noX_noA68_matched_allele,
            use_replay = True,
            **pepcl_base_params_con,
)
training_stats=pepcl.train()

[PepCL] Auto-calibrated sampling:
  MS: alleles=44, A_eff=19.80, balance=0.450, target_pos/cycle=1536, q_pos median=28.0 [18, 131], reuse median=0.0080, reuse max=0.0323, neg_ratio=1.0
  FT: alleles=49, A_eff=29.05, balance=0.593, target_pos/cycle=1536, q_pos median=28.0 [6, 69], reuse median=0.2478, reuse max=1.0000, neg_ratio=1.0



PepCL: 100%|██████████| 200/200 [02:37<00:00,  1.27it/s, MS=0.033007, FT=0.057715, DR=0.023094]


CPU times: user 2min 44s, sys: 23.1 s, total: 3min 7s
Wall time: 3min 4s


In [ ]:
# copy over negatives
def copy_domain0_negs_to_domain1(df, allele_col="allele", label_col="label", 
                                   seq_col="seq", domain_col="domain_id"):
    d0_negs = df[(df[domain_col] == 0) & (df[label_col] == 0)].copy()
    d0_negs[domain_col] = 1
    return pd.concat([df, d0_negs], ignore_index=True)

ms_escape_seq_train_data_noX_noA68_matched_allele_did0_negs_in_did1=copy_domain0_negs_to_domain1(ms_escape_seq_train_data_noX_noA68_matched_allele, allele_col="allele", label_col="label", seq_col="seq", domain_col="domain_id")
print_data_stats(ms_escape_seq_train_data_noX_noA68_matched_allele_did0_negs_in_did1)

Total: 24,367,624
Pos: 248,092 | 1.02%
Neg: 24,119,532 | 98.98%
N alleles: 49


In [ ]:
# now with the domain id 0 negatives copied over to domain id 1
escape_seq_neoant_ft_w_more_negs_cls = copy.deepcopy(mhcprime) # classification
escape_seq_neoant_pepcl_w_more_negs_cls = copy.deepcopy(mhcprime) # classification

In [ ]:
%%time
# run updated pepcl function
pepcl=PepCL(
            model=escape_seq_neoant_ft_w_more_negs_cls,
            train_df=ms_escape_seq_train_data_noX_noA68_matched_allele_did0_negs_in_did1,
            use_replay = False,
            **pepcl_base_params_cls,
)
training_stats=pepcl.train()

[PepCL] Auto-calibrated sampling:
  MS: alleles=44, A_eff=19.80, balance=0.450, target_pos/cycle=1536, q_pos median=28.0 [18, 131], reuse median=0.0080, reuse max=0.0323, neg_ratio=1.0
  FT: alleles=49, A_eff=29.05, balance=0.593, target_pos/cycle=1536, q_pos median=28.0 [6, 69], reuse median=0.2478, reuse max=1.0000, neg_ratio=1.0



PepCL: 100%|██████████| 200/200 [01:08<00:00,  2.93it/s, MS=0.000000, FT=0.000614, DR=0.000000]


CPU times: user 1min 30s, sys: 20.3 s, total: 1min 51s
Wall time: 1min 50s


In [ ]:
%%time
# run updated pepcl function
pepcl=PepCL(
            model=escape_seq_neoant_pepcl_w_more_negs_cls,
            train_df=ms_escape_seq_train_data_noX_noA68_matched_allele_did0_negs_in_did1,
            use_replay = True,
            **pepcl_base_params_cls,
)
training_stats=pepcl.train()

[PepCL] Auto-calibrated sampling:
  MS: alleles=44, A_eff=19.80, balance=0.450, target_pos/cycle=1536, q_pos median=28.0 [18, 131], reuse median=0.0080, reuse max=0.0323, neg_ratio=1.0
  FT: alleles=49, A_eff=29.05, balance=0.593, target_pos/cycle=1536, q_pos median=28.0 [6, 69], reuse median=0.2478, reuse max=1.0000, neg_ratio=1.0



PepCL: 100%|██████████| 200/200 [02:37<00:00,  1.27it/s, MS=0.009084, FT=0.001191, DR=0.004404]


CPU times: user 2min 55s, sys: 28.8 s, total: 3min 24s
Wall time: 3min 21s


In [ ]:
%%time
"""
Save models
"""
# save_models=True
save_models=False

models_to_save={

    # updated models
    # "escape_seq_neoant_ft_cls": escape_seq_neoant_ft_cls,
    # "escape_seq_neoant_pepcl_cls": escape_seq_neoant_pepcl_cls,
    # "escape_seq_neoant_ft_con": escape_seq_neoant_ft_con,
    # "escape_seq_neoant_pepcl_con": escape_seq_neoant_pepcl_con,

    "escape_seq_neoant_ft_w_more_negs_cls": escape_seq_neoant_ft_w_more_negs_cls,
    "escape_seq_neoant_pepcl_w_more_negs_cls": escape_seq_neoant_pepcl_w_more_negs_cls,

}

escape_seq_model_checkpoints_dir = MODEL_CHECKPOINTS_DIR / "ESCAPE_seq_Models"
os.makedirs(escape_seq_model_checkpoints_dir, exist_ok=True)

if save_models:
    for model_name, model in models_to_save.items():
        print(model_name)
        torch.save(model.state_dict(), f"{escape_seq_model_checkpoints_dir}/{model_name}.pt")

escape_seq_neoant_ft_w_more_negs_cls
escape_seq_neoant_pepcl_w_more_negs_cls
CPU times: user 207 ms, sys: 224 ms, total: 431 ms
Wall time: 426 ms


#### Inference

In [ ]:
# set models
base_model=copy.deepcopy(mhcprime)

base_and_ft_models={

    # base model
    # "mhcprime": base_model,

    # updated models
    # "escape_seq_neoant_ft_cls": escape_seq_neoant_ft_cls,
    # "escape_seq_neoant_pepcl_cls": escape_seq_neoant_pepcl_cls,
    # "escape_seq_neoant_ft_con": escape_seq_neoant_ft_con,
    # "escape_seq_neoant_pepcl_con": escape_seq_neoant_pepcl_con,

    "escape_seq_neoant_ft_w_more_negs_cls": escape_seq_neoant_ft_w_more_negs_cls,
    "escape_seq_neoant_pepcl_w_more_negs_cls": escape_seq_neoant_pepcl_w_more_negs_cls,

}


In [ ]:
%%time
# build cache for background set
build_caches=False # already built
if build_caches:
    data_to_run_path=f"{DATA_DIR}/chp_background_10k_per_allele_x_len_processed.pt"
    build_token_cache(chp_background_10k_per_allele_x_len_processed, data_to_run_path, tokenizer, use_domain_id=False)

CPU times: user 5 µs, sys: 1 µs, total: 6 µs
Wall time: 15 µs


In [ ]:
%%time
"""
Score background
"""
# chp background
data_to_run_path=f"{DATA_DIR}/chp_background_10k_per_allele_x_len_processed.pt"

# run_bg=False
run_bg=True

if run_bg:
    for model_name, model in base_and_ft_models.items():
        print(model_name)
        model.eval()
        with torch.no_grad():
            num_workers=16
            output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=num_workers, compile_model=False, use_domain_id=False)

        chp_background_10k_per_allele_x_len_processed[model_name]=output_scores.detach().cpu().tolist() 

escape_seq_neoant_ft_w_more_negs_cls
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 4484/4484 [09:04<00:00,  8.24it/s]


escape_seq_neoant_pepcl_w_more_negs_cls
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 4484/4484 [09:04<00:00,  8.23it/s]


CPU times: user 18min 27s, sys: 47.5 s, total: 19min 14s
Wall time: 18min 26s


In [ ]:
%%time
"""
Save ranking DF with scores
"""
# chp_background_10k_per_allele_x_len_processed.to_csv(f"{DATA_DIR}/chp_background_10k_per_allele_x_len_processed.csv.gz")

CPU times: user 5min 41s, sys: 2.27 s, total: 5min 43s
Wall time: 5min 43s


In [ ]:
# add base model (if not already added)
base_and_ft_models["mhcprime"]=base_model

In [ ]:
%%time
# escapeseq_reported_neoantigens_expanded
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(escapeseq_reported_neoantigens_expanded, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    escapeseq_reported_neoantigens_expanded[model_name]=output_scores.detach().cpu().tolist()

225it [00:00, 17627.73it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/dataset_cache.pt  (0.1 MB)
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1/1 [00:06<00:00,  6.26s/it]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1/1 [00:04<00:00,  4.31s/it]


CPU times: user 196 ms, sys: 10.5 s, total: 10.7 s
Wall time: 12.7 s


In [ ]:
# delete dataset cache
data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
os.remove(data_to_run_path)

In [ ]:
"""
Save reported neoantigens df
"""
# escapeseq_reported_neoantigens_expanded.to_csv(f"{escape_seq_datasets_dir}/escapeseq_reported_neoantigens_expanded.csv")

In [ ]:
%%time
# get rankings
escapeseq_reported_neoantigens_expanded["seq_len"]=escapeseq_reported_neoantigens_expanded.seq.str.replace("$", "").apply(len)
score_cols=["mhcprime", "escape_seq_neoant_ft_cls", "escape_seq_neoant_pepcl_cls", "escape_seq_neoant_ft_con", "escape_seq_neoant_pepcl_con"]
score_cols+=["escape_seq_neoant_ft_w_more_negs_cls", "escape_seq_neoant_pepcl_w_more_negs_cls"]
escapeseq_reported_neoantigens_expanded = add_background_percentile_ranks(
    test_df=escapeseq_reported_neoantigens_expanded,
    background_df=chp_background_10k_per_allele_x_len_processed,
    allele_col="allele",
    seq_len_col="seq_len",
    score_cols=score_cols,
    rank_mode="global",
    suffix="_rank_r_chp",
    reverse=True,
    copy=True,
)

Ranking score columns: 100%|██████████| 7/7 [00:02<00:00,  2.70it/s]

CPU times: user 2 s, sys: 913 ms, total: 2.91 s
Wall time: 2.89 s


In [ ]:
"""
Save reported neoantigens df
"""
# escapeseq_reported_neoantigens_expanded.to_csv(f"{escape_seq_datasets_dir}/escapeseq_reported_neoantigens_expanded.csv")

In [ ]:
# set models
base_model=copy.deepcopy(mhcprime)

base_and_ft_models={

    # base model
    "mhcprime": base_model,

    # updated models
    "escape_seq_neoant_ft_cls": escape_seq_neoant_ft_cls,
    "escape_seq_neoant_pepcl_cls": escape_seq_neoant_pepcl_cls,
    "escape_seq_neoant_ft_con": escape_seq_neoant_ft_con,
    "escape_seq_neoant_pepcl_con": escape_seq_neoant_pepcl_con,

    "escape_seq_neoant_ft_w_more_negs_cls": escape_seq_neoant_ft_w_more_negs_cls,
    "escape_seq_neoant_pepcl_w_more_negs_cls": escape_seq_neoant_pepcl_w_more_negs_cls,

}

In [ ]:
%%time
# ligand_mhc_atlas_cancer_test_set_for_escape_seq
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(ligand_mhc_atlas_cancer_test_set_for_escape_seq, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    ligand_mhc_atlas_cancer_test_set_for_escape_seq[model_name]=output_scores.detach().cpu().tolist()

29674it [00:01, 24269.62it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/dataset_cache.pt  (8.1 MB)
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 10/10 [00:06<00:00,  1.43it/s]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 10/10 [00:07<00:00,  1.26it/s]


...

Getting data
Loading data
Setting up model
Running inference


100%|██████████| 10/10 [00:06<00:00,  1.44it/s]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 10/10 [00:07<00:00,  1.31it/s]


CPU times: user 10.1 s, sys: 42.7 s, total: 52.8 s
Wall time: 1min


In [ ]:
# delete dataset cache
data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
os.remove(data_to_run_path)

In [ ]:
"""
Save
"""
ligand_mhc_atlas_output_dir=f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Ligand_MHC_Atlas"
# ligand_mhc_atlas_cancer_test_set_for_escape_seq.to_csv(f"{ligand_mhc_atlas_output_dir}/ligand_mhc_atlas_cancer_test_set_for_escape_seq.csv")

In [ ]:
%%time
# get ranks
ligand_mhc_atlas_cancer_test_set_for_escape_seq["seq_len"]=ligand_mhc_atlas_cancer_test_set_for_escape_seq.seq.str.replace("$", "").apply(len)

score_cols=["mhcprime", "escape_seq_neoant_ft_cls", "escape_seq_neoant_pepcl_cls", "escape_seq_neoant_ft_con", "escape_seq_neoant_pepcl_con"]
score_cols+=["escape_seq_neoant_ft_w_more_negs_cls", "escape_seq_neoant_pepcl_w_more_negs_cls"]

ligand_mhc_atlas_cancer_test_set_for_escape_seq = add_background_percentile_ranks(
    test_df=ligand_mhc_atlas_cancer_test_set_for_escape_seq,
    background_df=chp_background_10k_per_allele_x_len_processed,
    allele_col="allele",
    seq_len_col="seq_len",
    score_cols=score_cols,
    rank_mode="global",
    suffix="_rank_r_chp",
    reverse=True,
    copy=True,
)


Ranking score columns: 100%|██████████| 7/7 [00:02<00:00,  3.49it/s]

CPU times: user 1.73 s, sys: 552 ms, total: 2.28 s
Wall time: 2.26 s


In [ ]:
"""
Save
"""
ligand_mhc_atlas_output_dir=f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Ligand_MHC_Atlas"
# ligand_mhc_atlas_cancer_test_set_for_escape_seq.to_csv(f"{ligand_mhc_atlas_output_dir}/ligand_mhc_atlas_cancer_test_set_for_escape_seq.csv")

### Differential mutation results inference

In [ ]:
%%time
# process gene holdout results
central_ft_dir=f"{DATA_DIR}/ESCAPE_seq_gene_holdout_experiment"
output_df_dir="inference_dfs"
output_df_dir_post_inference="inference_dfs_updated"
os.makedirs(f"{central_ft_dir}/{output_df_dir_post_inference}", exist_ok=True)

genes_to_run=['JAK2', 'MAX', 'MPL', 'MED12', 'TP53', 'SRSF2', 'NOTCH2', 'EZH2', 'AKT1', 'IDH1', 'ATM', 'MYD88', 'EGFR', 'FGFR3', 'PTEN', 'SF3B1', 'CREBBP', 'MYC', 'FBXW7', 'BRAF', 'RET', 'SMAD4', 'ERBB2', 'GNAS', 'DNMT3A', 'ALK', 'KIAA1549', 'EWSR1']

# add genes from gene groups
gene_groups={
    "KRAS": ["KRAS", "KRAS-NRAS", "KRAS-KRAS-NRAS-NRAS-HRAS-HRAS", "KRAS-HRAS", "KRAS-NRAS-HRAS", "HRAS", "NRAS-HRAS"], 
    "FLT3": ["FLT3", "KIT", "PDGFRA", "FLT3-KIT-PDGFRA", "FLT3-PDGFRA"],
    "IDH2": ["IDH2", "IDH2-IDH2"],
    "PIK3CA": ["PIK3CA", "PIK3CA-PIK3CA"],
    "CTNNB1": ["CTNNB1", "CTNNB1-CTNNB1"],
}

genes_to_run+=list(gene_groups.keys())

allele_ap_matrix_wt_neg_mut_pos=pd.DataFrame()
allele_ap_matrix_wt_pos_mut_neg=pd.DataFrame()

full_spearmans=pd.DataFrame()
wt_spearmans=pd.DataFrame()
mut_spearmans=pd.DataFrame()
wt_mut_gene_diff_noX_spearmans=pd.DataFrame()

for gene in tqdm(genes_to_run):

    mut_wt_df_gene=pd.read_csv(f"{central_ft_dir}/{output_df_dir}/mut_wt_df_gene_{gene}.csv")

    mut_wt_df_gene["name_allele"]=[f"{n}_{a}" for n, a in zip(mut_wt_df_gene.name2, mut_wt_df_gene.allele)]

    model_name_ft=f"model_ft_{gene}"
    model_name_pepcl=f"model_pepcl_{gene}"    

    # inference
    escape_seq_model_dir_ft = MODEL_CHECKPOINTS_DIR / "ESCAPE_seq_gene_holdout_experiment" / "FT"
    escape_seq_model_dir_pepcl = MODEL_CHECKPOINTS_DIR / "ESCAPE_seq_gene_holdout_experiment" / "PepCL"

    model_ft=init_model(base_model_params, print_check=False, print_params=False)
    model_ft.load_state_dict(torch.load(f"{escape_seq_model_dir_ft}/{model_name_ft}.pt"))

    model_pepcl=init_model(base_model_params, print_check=False, print_params=False)
    model_pepcl.load_state_dict(torch.load(f"{escape_seq_model_dir_pepcl}/{model_name_pepcl}.pt"))

    base_model=copy.deepcopy(mhcprime)
    base_and_ft_models={
        "model_base": base_model,
        "model_ft": model_ft,
        "model_pepcl": model_pepcl,
    }

    # mut_wt_df_gene
    build_caches=True

    if build_caches:
        data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

        build_token_cache(mut_wt_df_gene, data_to_run_path, tokenizer, use_domain_id=False)

    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
    for model_name, model in base_and_ft_models.items():
        model.eval()
        with torch.no_grad():
            output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=2, compile_model=False, use_domain_id=False)

        mut_wt_df_gene[model_name]=output_scores.detach().cpu().tolist()

    mut_wt_df_gene.to_csv(f"{central_ft_dir}/{output_df_dir_post_inference}/mut_wt_df_gene_{gene}.csv")

# delete dataset cache
data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
os.remove(data_to_run_path)

  0%|          | 0/33 [00:00<?, ?it/s]

[FeatureProjectionEncoder] Warning: no features found for tokens: ['$', '*', '<BOS>']
[FeatureProjectionEncoder] Warning: no features found for tokens: ['$', '*', '<BOS>']


6300it [00:00, 23931.57it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/dataset_cache.pt  (1.7 MB)
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 3/3 [00:00<00:00,  6.60it/s]


...

Getting data
Loading data
Setting up model
Running inference


100%|██████████| 2/2 [00:00<00:00,  2.73it/s]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 33/33 [01:33<00:00,  2.84s/it]

CPU times: user 12min 29s, sys: 27.8 s, total: 12min 57s
Wall time: 1min 33s
